In [1]:
import pandas as pd
#import os
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
#from bs4 import BeautifulSoup
import re
#from collections import defaultdict
from tqdm import tqdm
#from datetime import datetime
import random

data_folder = '/raid/deallab/SF_RAG_Data/ASQA'
# data_folder = '../data'

tokenizer = AutoTokenizer.from_pretrained(
   "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture # "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp"
)

token_length = 1024 # adjust to maximal token length
document_limit = 10000
dataset = 'dev' # test

# restricting legth (make room for cls token and paragraph seperators) 
TOK_LEN = token_length - 24

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # read  data
# df = pd.read_parquet(f'{data_folder}/train.parquet')

# for col in df.columns:
#     print(col,':')
#     print(df.loc[1, col], '\n')

In [3]:
#helper
def get_wikipage_title(url):
    return url.split('/')[-1]

def split_list(a, max_len, o):
    o_len = len(a) + ceil(len(a)/max_len) * o 
    n = ceil(o_len/ max_len)
    k, m = divmod(o_len, n)
    return (a[i*(k-o)+min(i, m):(i+1)*k -i*o+min(i+1, m)] for i in range(n))

# api calls 
def get_document(session, title):
    url = 'https://en.wikipedia.org/w/api.php?'
    headers = {'User-Agent': 'sf_rag/1.0; lassejantsch@knu.ac.kr)'}
    params = {
        'action': 'query',
        'prop': 'revisions|extracts',
        'titles': title,
        'rvstart': '2020-02-01T00:00:00Z',
        'rvlimit': '1',
        'rvdir': 'older',
        'rvprop': 'ids|timestamp',
        'rvslots': 'main',
        'formatversion':'2',
        'format': 'json'
    }
    response = {}

    # get document text
    try:
        res =requests.get(url + '&'.join([f'{k}={v}' for k,v in params.items()]), headers= headers)
        
        #parse docuements
        res_json = res.json()
        response = [v for k,v in res_json['query']['pages'][0].items() if k in ['extract','title']]
    except:
        print(res.status_code, res.url)
    
    return response


In [4]:

class DocElement():
    def __init__(self, text, tokens, level):
        self.text = text
        self.level = int(level)
        self.tokens = tokens
        self.length = len(tokens)
        
    def __len__(self):
        return self.length
    
    def __str__(self):
        return "\t"* (self.level+1) + self.text

    def get_text(self):
        return self.text
    
    def split_doc(self, max_tokens, title):
        raise NotImplementedError

class DocSection(DocElement):
    def __init__(self, title, tokens, level):
        super().__init__(title, tokens, level)
        self.title = self.text
        self.content = []
        self.has_subsection = False
        
    
    def __str__(self):
        print_str = ''.join([str(el) for el in self.content])
        return '{0}{1}'.format("\t"*self.level + self.title,print_str)
        
        
    def update_length(self):
        len_of_content = sum([len(el) for el in self.content])
        self.length = len(self.tokens) + len_of_content
        # print(self.title,len(self.tokens), len_of_content, self.length)
    
    def get_text(self):
        text = self.title
        for element in self.content:
            text += element.get_text()
        return text
    
    def append(self, content, tokens, level, type):
        if type == 'sec':
            if level-1 == self.level:
                self.content.append(DocSection(content, tokens, level))
                self.has_subsection=True
            else:
                self.content[-1].append(content, tokens, level, type)                
        elif type=='par':
            if self.has_subsection:
                self.content[-1].append(content, tokens, level, type)
            else:
                self.content.append(DocElement(content, tokens, level))
        self.update_length()
    
    def split_doc(self, max_tokens, title=''):
        if title == '':
            title = re.search(r'#+ (.*?) #+', self.title).group(1)
        else:
            title = '{0}/{1}'.format(title, re.search(r'#+ (.*?) #+', self.title).group(1))
        title_str = 'Document: {0}\n\n'.format(title)
        title_str_len = len(tokenizer.encode(title_str, add_special_tokens=False))

        # if whole doc fits into max tokens
        if self.length + title_str_len <= max_tokens:
            return [self.get_text()]
        
        # split if not
        splitted_doc_ls = []
        current_split= title_str
        current_split_len = title_str_len
        for element in self.content:
            if len(element) == 0: continue # continue if empty element
            if current_split_len + len(element) <= max_tokens:
                #print(self.level,current_split_len, len(element), 'added to current')
                current_split += element.get_text()
                current_split_len += len(element)
            elif len(element)+title_str_len <= max_tokens:
                #print(self.level, current_split_len, len(element), 'added to new')
                splitted_doc_ls.append(current_split)
                current_split = title_str + element.get_text()
                current_split_len = title_str_len + len(element)
            else:
                #print(self.level,current_split_len, len(element), 'recursion')
                if current_split_len > title_str_len: 
                    splitted_doc_ls.append(current_split)
                    current_split = title_str
                    current_split_len = title_str_len
                splitted_doc_ls.extend(element.split_doc(max_tokens, title))
        if current_split_len > title_str_len: splitted_doc_ls.append(current_split)
        return splitted_doc_ls
                
                 

class Document(DocSection):
    def __init__(self, title, tokenizer, max_len = 1024):
        self.tokenizer = tokenizer
        self.last_level = 0
        self.max_len = max_len
        super().__init__(title, self.tokenizer.encode(title, add_special_tokens=False), 0)
    
    def add(self, content, level = None):
        if not level:
            tokens = self.tokenizer.encode(content, add_special_tokens=False)
            if len(tokens) < self.max_len - 100:
                self.append(content, tokens, self.last_level, 'par')
            else:
                token_ls = split_list(tokens, self.max_len -100, 200)
        else:
            self.append(content, self.tokenizer.encode(content, add_special_tokens=False), level, 'sec')
            self.last_level = level
        

In [5]:
# remove all html tags
def clean_html_tags(string):
    return re.sub('<[^>]*>', '', string).strip()

#pars unordered lists to text
def get_ul(ul):
    list_items = re.findall(r'<li[^>]*>(.*?)</li>', ul)
    return '\n'.join([f'* {item}' for item in list_items if item.strip()])
# pars ordered list to text
def get_ol(ol):
    list_items = re.findall(r'<li[^>]*>(.*?)</li>', ol)
    return '\n'.join([f'{id+1}. {item}' for id, item in enumerate(list_items) if item.strip()])

def get_heading(h):
    heading_type = int(re.search(r'^<h(\d)', h).group(1))
    heading_text = clean_html_tags(h.replace('\n',''))
    return f"{'#'*heading_type} {heading_text if heading_text else 'Unknown'} {'#'*heading_type}\n", heading_type -1

def parse_document(title, doc_str):
    doc_str = ' '.join(doc_str.split())
    doc_ls = re.findall(r'<p.*?</p>|<h\d.*?</h\d>|<ul.*?</ul>|<ol.*?</ol>', doc_str)
    
    parsed_doc = Document(f'# {title} #\n', tokenizer)
    for entry in doc_ls:
        if re.match(r'^<h', entry):
            if clean_html_tags(entry) in ['See also', 'References']: break # break condition when main article is over
            entry_text, level = get_heading(entry)
            parsed_doc.add(entry_text, level)
        if re.match(r'^<p', entry):
            clean_entry = clean_html_tags(entry.replace('\n',''))
            parsed_doc.add(clean_entry + '\n')
        elif re.match(r'^<ul', entry):
            clean_entry = clean_html_tags(get_ul(entry))
            parsed_doc.add(clean_entry)
        elif re.match(r'^<ol', entry):
            clean_entry = clean_html_tags(get_ol(entry))
            parsed_doc.add(clean_entry)
    
    return parsed_doc.split_doc(1024)
    

In [6]:
# # create embedding document dataset.
# train_embeddin_easy = pd.DataFrame(columns=['question', 'text'])
# train_embedding_hard = pd.DataFrame(columns=['question', 'text_pos','text_neg'])
# embedding_easy_path={data_folder}/train/train_embeddin_easy.csv'
# embedding_hard_path={data_folder}/train/train_embedding_hard.csv'
# train_embeddin_easy.to_csv(embedding_easy_path, index=False)
# train_embedding_hard.to_csv(embedding_hard_path, index=False)

# # create evidence / qq data for qq training
# evidence_train = pd.DataFrame(columns=['text'])
# evidence_train_path={data_folder}/train/evidence_train.csv'
# evidence_train.to_csv(evidence_train_path, index=False)

# #creating question question pairs for retrival training (not implemented)
# follow_up_train = pd.DataFrame(columns=['question', 'follow_up_questions'])
# follow_up_train_path={data_folder}/train/follow_up_train.csv'
# follow_up_train.to_csv(follow_up_train_path, index=False)

# # init params
# session = requests.Session() # initiate session
# added_evidence = set() # empty evidence set

# document_limit = 10000

# for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
#         try:
#                 #break when document limit is reached 
#                 if idx == document_limit: break
                
#                 #extract information from sample
#                 sample_id = row['sample_id']
#                 evidence_title_mapping = {evidence['title']:get_wikipage_title(evidence['url']) for evidence in row['wikipages']}
#                 base_question = row['ambiguous_question']
#                 follow_up_qa_mapping = {qa['question']:qa['short_answers'] for qa in row['qa_pairs']}
#                 follow_up_qe_mapping = {qa['question']:qa['wikipage'] for qa in row['qa_pairs'] if qa['wikipage']}
#                 long_answers = [answ['long_answer'] for answ in row['annotations']]
                
#                 # crawl documents
#                 evidence_docs = {}
#                 for title, url_title in evidence_title_mapping.items():
#                         _title, text = get_document(session, url_title)
#                         evidence_docs[title] = parse_document(title, text)
                        
#                 #fill train embedding easy
#                 for title, docs in evidence_docs.items():
#                         for doc in docs:
#                                 train_embeddin_easy.loc[len(train_embeddin_easy)] = [base_question, doc]
        
#                 # fill train embeddin hard
#                 for question, title in follow_up_qe_mapping.items():
#                         if title not in evidence_docs or len(evidence_docs) < 2: continue
#                         rel_docs = evidence_docs[title]
#                         other_docs = [doc for t, docs in evidence_docs.items() if t != title for doc in docs ]
#                         for doc in rel_docs:
#                                 train_embedding_hard.loc[len(train_embedding_hard)] = [question, doc, random.choices(other_docs, k=1)[0]]        
#                                 train_embedding_hard.loc[len(train_embedding_hard)] = [question, doc, random.choices(other_docs, k=1)[0]]     
                
#                 # fill train evidence
#                 for title, docs in evidence_docs.items():
#                         if title in added_evidence: continue
#                         added_evidence.add(title)
#                         for doc in docs:
#                                 evidence_train.loc[len(evidence_train)] = [doc]
                
#                 # fill follow up question train
#                 for _ in range(len(follow_up_qa_mapping)-1):
#                         questions = random.sample(list(follow_up_qa_mapping.keys()), len(follow_up_qa_mapping))
#                         follow_up_questions = '\n'.join([f'### {question}' for question in questions])
#                         follow_up_train.loc[len(follow_up_train)] = [base_question, follow_up_questions]
#         except:
#                 print('something went wrong')

#         #save dataframes after 100 iterations
#         if (idx + 1 )% 100 == 0:
#                 #save
#                 train_embeddin_easy.to_csv(embedding_easy_path, mode='a', header=False, index=False)
#                 train_embedding_hard.to_csv(embedding_hard_path, mode='a', header=False, index=False)
#                 evidence_train.to_csv(evidence_train_path, mode='a', header=False, index=False)
#                 follow_up_train.to_csv(follow_up_train_path, mode='a', header=False, index=False)
                
#                 #empty
#                 train_embeddin_easy = pd.DataFrame(columns=['question', 'text'])
#                 train_embedding_hard = pd.DataFrame(columns=['question', 'text_pos','text_neg'])
#                 evidence_train = pd.DataFrame(columns=['text'])
#                 follow_up_train = pd.DataFrame(columns=['question', 'follow_up_questions'])

        
# #save
# train_embeddin_easy.to_csv(embedding_easy_path, mode='a', header=False, index=False)
# train_embedding_hard.to_csv(embedding_hard_path, mode='a', header=False, index=False)
# evidence_train.to_csv(evidence_train_path, mode='a', header=False, index=False)
# follow_up_train.to_csv(follow_up_train_path, mode='a', header=False, index=False)

In [7]:
# read  data
df = pd.read_parquet(f'{data_folder}/dev.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[0, col], '\n')

ambiguous_question :
Who has the highest goals in world football? 

qa_pairs :
[{'context': 'No context provided', 'question': "Who has the highest goals in men's world international football?", 'short_answers': array(['Daei', 'Ali Daei'], dtype=object), 'wikipage': None}
 {'context': 'No context provided', 'question': "Who has the highest goals all-time in men's football?", 'short_answers': array(['Bican', 'Josef Bican'], dtype=object), 'wikipage': None}
 {'context': 'The first player to reach 100 international goals was Italian Elisabetta Vignotto. Abby Wambach scored 100 goals in 9 years, while Christine Sinclair reached the milestone in just under 10 years while Mia Hamm is the youngest player to score 100 international goals at the age of 26 years 185 days. Most played exclusively in the forward position, with Kristine Lilly and Michelle Akers having also played as midfielder. All players scored at a high average rate of more than one goal every three matches. International goals 

In [8]:
### solve plroblems
    # 1. array([sort_answ])
    # 2. [f, o, l, l, o, w, u, p, q, u, e, s, t, i, o, n]

# create embedding document dataset.
evidence_test = pd.DataFrame(columns=['sample_id', 'title', 'text']) # init df
evidence_test_path = f'{data_folder}/test/evidence_test_eval.csv'
evidence_test.to_csv(evidence_test_path, index=False)

qa_test = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])
qa_test_path = f'{data_folder}/test/qa_test.csv'
qa_test.to_csv(qa_test_path, index=False)

# init params
session = requests.Session() # initiate session
added_evidence = set() # empty evidence set

document_limit = 1000

for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    try:
        #break when document limit is reached 
        if idx == document_limit: break
        
        #extract information from sample
        sample_id = row['sample_id']
        evidence_title_mapping = {evidence['title']:get_wikipage_title(evidence['url']) for evidence in row['wikipages']}
        base_question = row['ambiguous_question']
        follow_up_questions = [qa['question'] for qa in row['qa_pairs']]
        short_answers = [qa['short_answers'].tolist() for qa in row['qa_pairs']]
        long_answers = [answ['long_answer'] for answ in row['annotations']]
        
        # crawl documents
        evidence_docs = {}
        for title, url_title in evidence_title_mapping.items():
                _title, text = get_document(session, url_title)
                evidence_docs[title] = parse_document(title, text)
            
        # fill test evidence
        for title, docs in evidence_docs.items():
                if title in added_evidence: continue
                added_evidence.add(title)
                for doc in docs:
                        # evidence_test.loc[len(evidence_test)] = [doc]
                        evidence_test.loc[len(evidence_test)] = [sample_id, title, doc]# what I (djk) fixed to match query with documents
        
        # fill qa test
        qa_test.loc[len(qa_test)] = [uuid4(), sample_id, base_question, follow_up_questions, long_answers, short_answers]
    except Exception as  e:
        print(f'something went wrong: {e}')
        
    if (idx + 1 )% 100 == 0:
        evidence_test.to_csv(evidence_test_path, mode='a', header=False, index=False)
        qa_test.to_csv(qa_test_path,mode='a', header=False, index=False)
        # evidence_test = pd.DataFrame(columns=['text'])
        evidence_test = pd.DataFrame(columns=['sample_id', 'title', 'text'])
        qa_test = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])
        

evidence_test.to_csv(evidence_test_path, mode='a', header=False, index=False)
qa_test.to_csv(qa_test_path,mode='a', header=False, index=False)

  2%|▏         | 20/948 [00:27<18:37,  1.20s/it]

something went wrong: 'DocElement' object has no attribute 'append'


 15%|█▌        | 145/948 [03:28<15:41,  1.17s/it]

something went wrong: not enough values to unpack (expected 2, got 1)


 16%|█▌        | 154/948 [03:41<20:43,  1.57s/it]

...

 75%|███████▍  | 709/948 [19:18<07:09,  1.80s/it]

something went wrong: not enough values to unpack (expected 2, got 1)


 90%|████████▉ | 852/948 [23:50<02:56,  1.84s/it]

something went wrong: 'NoneType' object has no attribute 'split'


100%|██████████| 948/948 [26:45<00:00,  1.69s/it]


In [20]:
evidence_docs

{'Sign of the Times (Harry Styles song)': ['Document: Sign of the Times (Harry Styles song)\n\n\n"Sign of the Times" is the debut solo single by English singer-songwriter Harry Styles from his self-titled debut studio album. Released on 7 April 2017 by Columbia Records, it was first written by Jeff Bhasker, Mitch Rowland, Ryan Nasci, Alex Salibian, while Styles gets writing credits for contributing. It was produced by Bhasker and co produced by Salibian and Johnson. Musically, it was described by critics as a pop rock and soft rock ballad. Its accompanying music video was released on 8 May 2017.\n"Sign of the Times" reached number one on the UK charts and number four in the United States. In 2018, the single won a BMI Pop Award, and the video won a Brit Award for British Artist Video of the Year. In 2021, Rolling Stone placed it at number 428 on its list of The 500 Greatest Songs of All Time.\n## Background and release ##\nRumours about Styles embarking on a solo career sparked in 2015

In [11]:
import pandas as pd
evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test_eval.csv'

df1 = pd.read_csv(evidence_test_path)
len(df1)

11699

In [16]:
df1['text'][0]

'# International Federation of Football History & Statistics #\n'

In [12]:
import pandas as pd
evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'

df2 = pd.read_csv(evidence_test_path)
len(df2)

21586

In [19]:
df2.head(20)

,text
0,Document: International Federation of Football...
1,Document: International Federation of Football...
2,Document: International Federation of Football...
3,Document: International Federation of Football...
4,Document: International Federation of Football...
5,Document: International Federation of Football...
6,Document: International Federation of Football...
7,Document: International Federation of Football...
8,Document: International Federation of Football...
9,Document: International Federation of Football...
